In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from PIL import Image, ImageDraw

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.build import build_finetune_loader, build_finetune_model
from src.finetune.checkpoint import load_trainable_state

CONFIG_PATH = ROOT / 'config/finetune_test.yaml'
LORA_PATH = None
BATCH_SIZE = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = yaml.safe_load(CONFIG_PATH.read_text())
model_config = dict(config['model'])
model_config['path'] = str(ROOT / model_config['path'])
model_config['device'] = device
model = build_finetune_model(model_config)
if LORA_PATH is not None:
    checkpoint = torch.load(LORA_PATH, map_location='cpu', weights_only=False)
    load_trainable_state(model, checkpoint['model'])
model.eval()

loader_config = dict(config['data']['valid'])
loader_config['batch_size'] = BATCH_SIZE
loader_config['num_workers'] = 0
loader = build_finetune_loader(loader_config, train=False)


In [ ]:
def to_image(tensor):
    array = tensor.permute(1, 2, 0).numpy()
    return Image.fromarray(np.clip((array + 1) * 127.5, 0, 255).astype('uint8'))

def overlay(image, mask, color):
    mask = Image.fromarray((mask * 255).astype('uint8')).resize(image.size)
    layer = Image.new('RGB', image.size, color)
    return Image.composite(Image.blend(image, layer, 0.45), image, mask)

def draw_prompt(image, prompt):
    image = image.copy()
    draw = ImageDraw.Draw(image)
    if prompt['points'] is not None:
        for x, y in prompt['points']:
            draw.ellipse((x - 10, y - 10, x + 10, y + 10), fill='cyan')
    if prompt['box'] is not None:
        draw.rectangle(tuple(prompt['box']), outline='cyan', width=5)
    return image

def title(image, text):
    image = image.copy()
    ImageDraw.Draw(image).text((10, 10), text, fill='white', stroke_width=2, stroke_fill='black')
    return image.resize((420, 420))

def show_batch(batch, out):
    rows = []
    logits = F.interpolate(out['mask_logits'], size=batch['target'].shape[-2:], mode='bilinear', align_corners=False)
    masks = (logits.sigmoid()[:, 0] >= 0.5).numpy()
    scores = out['iou_scores'][:, 0].numpy()
    classes = out['class_logits'][:, 0].sigmoid().numpy()
    for index in range(len(masks)):
        image = to_image(batch['image'][index])
        prompt = batch['prompt'][index]
        expected = batch['target'][index, 0].numpy()
        label = batch['label_target'][index].int().tolist()
        cond = int(batch['cond'][index])
        intersection = np.logical_and(masks[index], expected > 0.5).sum()
        union = np.logical_or(masks[index], expected > 0.5).sum()
        actual_iou = intersection / max(union, 1)
        rows.append((
            title(draw_prompt(image, prompt), f"input  {prompt['type']}  cond={cond}"),
            title(overlay(image, expected, (40, 220, 80)), f'answer  class={label}'),
            title(overlay(image, masks[index], (255, 40, 160)), f'predict  iou={actual_iou:.3f}/{scores[index]:.3f}  class={classes[index].round(3)}'),
        ))
    sheet = Image.new('RGB', (1260, 420 * len(rows)), 'white')
    for row_index, row in enumerate(rows):
        for column_index, panel in enumerate(row):
            sheet.paste(panel, (420 * column_index, 420 * row_index))
    return sheet


In [ ]:
batch = next(loader)
model_batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}
with torch.inference_mode(), torch.autocast(device.type, enabled=device.type == 'cuda'):
    out = model(model_batch)
out = {key: value.float().cpu() for key, value in out.items()}
show_batch(batch, out)
